In [1]:
# Define the function to create and configure the job application
def create_application():
    """
    Create and configure the job application 
    """

    company_name = input("Enter the company name: ") # Prompt the user to enter the company name
    role = input("Enter the role you are applying for: ") # Prompt the user to enter the role they are applying for
    application_status = input("Enter the application status (e.g., Applied, Interviewing, Offer, Rejected): ") # Prompt the user to enter the application status
    timeframe = input("Enter the date applied (YYYY-MM-DD): ") # Prompt the user to enter the date applied

    application = {
        "company": company_name,
        "role": role,
        "status": application_status,
        "date": timeframe,
    }
    return application

In [2]:
# Test the function by calling it and printing the result
create_application()

{'company': 'IBM', 'role': 'QA', 'status': 'Offer', 'date': '20216-01-07'}

In [3]:
# Initialize an empty list to store job applications
applications = []
applications.append(create_application()) # Call the function and add the returned application to the list
print(applications) # Print the list of job applications

[{'company': 'Intel', 'role': 'Technitian', 'status': 'Applied', 'date': '2026-06-14'}]


In [4]:
# Define a function to collect multiple job applications from the user
def collect_applications():

    applications = []

    while True:
        applications.append(create_application()) # Call the function and add the returned application to the list

        another = input("Add another application? (y/n): ") # Prompt the user to add another application
        if another.lower() != "y":
            break

    return applications

In [5]:
# Test the function by calling it and printing the result
my_applications = collect_applications()
print(my_applications)

[{'company': 'Teleperformance', 'role': 'Analyst', 'status': 'Interviewing', 'date': '2026-07-20'}]


In [6]:
# Define a function to list all job applications
def list_applications(applications):
    if not applications:
        print("No applications to show yet.")
        return

    # APPENDIX: Print the list of job applications in a user-friendly format
    print("\nYour job applications:")
    for number, app in enumerate(applications, start=1):
        print(f"{number}. {app['company']} — {app['role']} — {app['status']} — {app['date']}")

In [7]:
# Test the function by calling it and printing the result
list_applications(my_applications)


Your job applications:
1. Teleperformance — Analyst — Interviewing — 2026-07-20


In [8]:
# Install the python-docx library to work with Word documents
!pip install python-docx

import docx
print(docx.__version__)

1.2.0


In [9]:
# Define a function to save the job applications to a Word document
import docx
from datetime import datetime

def save_report(applications):

    doc = docx.Document()

    today = datetime.now().strftime("%Y-%m-%d")
    doc.add_heading("My job applications Report", 0)
    doc.add_paragraph(f"Report generated on {today}")

    # APPENDIX: Create a table with a header row
    table = doc.add_table(rows=1, cols=4)
    table.style = "Table Grid"

    hdr_cells = table.rows[0].cells # Add header row to the table
    hdr_cells[0].text = "Company"
    hdr_cells[1].text = "Role"
    hdr_cells[2].text = "Status"
    hdr_cells[3].text = "Date Applied"

    
    for app in applications: # # Add one row per application
        row_cells = table.add_row().cells
        row_cells[0].text = app["company"]
        row_cells[1].text = app["role"]
        row_cells[2].text = app["status"]
        row_cells[3].text = app["date"]

    filename = f"job_applications_{today}.docx"
    doc.save(filename)
    print(f"Report saved as {filename}")

    return filename # Test the function by calling it and printing the result

In [10]:
# Save the report to a Word document
save_report(my_applications)

Report saved as job_applications_2026-07-27.docx


'job_applications_2026-07-27.docx'

## Google API Authentication

This section uses the Gmail API to log in and send the report by email.
Authentication is handled via OAuth 2.0. The ** credentials.json ** file
(downloaded from Google Cloud) is read by `get_creds()`, and a `token.json`
is created on first login so the user does not log in every time.

Note: `credentials.json` and `token.json` are excluded from version control.

In [15]:
# Google API authentication
# Based on the Gmail API code provided in class (S. Weiss)
import os.path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


def get_creds():
    """Authenticate with Google and return valid credentials.

    Reads credentials.json, opens a browser for login on first run,
    and saves token.json so future runs log in automatically.
    """
    SCOPES = ["https://www.googleapis.com/auth/gmail.modify",
              "https://www.googleapis.com/auth/gmail.send"]

    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
        print("Have tokens!")

    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
            print("Refreshed token!")
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "credentials.json", SCOPES
            )
            creds = flow.run_local_server(port=0)
            print("Got new token")

    # Save the credentials for the next run
    with open("token.json", "w") as token:
        token.write(creds.to_json())

    return creds

In [ ]:
# Get credentials for Gmail API
creds = get_creds()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=406205782606-3eak3bhjdk2vqic4hfqj413e1rh8hbdh.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A54508%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.modify+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.send&state=5kSDpjWCpGW09qvbfwecZqmWerzoTA&code_challenge=Et1Rc2Molq6iyv_4pt29NKmG7jsQ9QtyHWOFJXP8hUg&code_challenge_method=S256&access_type=offline
Got new token


In [ ]:
# Gmail API code to send an email
import base64
from email.message import EmailMessage
# Based on the Gmail API code provided in class (S. Weiss)
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


def gmail_send_message(messageContent, subject, recipient, attachment=None):